# Data Standardization Notebook

This notebook applies minimal standardization to prepare the dataset for machine learning models.

## Overview
- **Input**: `features_derived.csv`
- **Output**: `features_standardized.csv`
- **Key Operations**:
  1. Validate data quality and formatting
  2. Convert data types appropriately
  3. Handle categorical variables
  4. Standardize binary/flag columns
  5. Ensure numeric columns have proper types

## 1. Setup and Configuration

### 1.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

### 1.2 Load Data

In [ ]:
# Define paths
current_user = os.getlogin()
DATA_DIR = Path(f"/home/{current_user}/local-share")
OUT_DIR = DATA_DIR / "processed"

# Load derived features
data = pd.read_csv(OUT_DIR / "features_derived.csv", sep=None, engine="python", encoding="utf-8-sig")
print(f"Data shape: {data.shape}")
data.head()

## 2. Data Quality Validation

### 2.1 Standardization Plan

In [ ]:
# =====================================================================
# MINIMAL STANDARDIZATION + VALIDATION PLAN  
# =====================================================================
# Goal: Apply basic treatment to ensure the dataset is:
#   • Safe for statistical analysis
#   • Consistent in SI units and formatting
#   • Free of weird numeric values (inf / huge magnitudes)
#   • Correctly typed for later modeling (object for categoricals)
#
# Workflow:
#   1. SI-Unit & Formatting Inspection
#   2. Weird-Value Detection (inf / huge values)
#   3. Data Type Standardization:
#      - Convert datetime columns
#      - Cast categorical columns to object dtype
#      - Force binary/flag columns to Int64 (0/1)
#      - Force continuous numeric columns to float
#      - Replace ±inf with NaN
# ===================================================================== 

### 2.2 Inspect Numeric Values for SI-Unit Consistency

In [ ]:
# Examine numeric values for consistency (excludes ID, object, datetime, binary columns)
print("🔍 Inspecting unique values in numeric columns...")

for col in data.columns:
    col_lower = col.lower()
    series = data[col]

    # Skip ID, object, datetime, and binary columns
    if ("id" in col_lower or 
        series.dtype == "object" or 
        pd.api.types.is_datetime64_any_dtype(series)):
        continue
    
    # Skip binary columns (only 0/1 values)
    unique_vals = series.dropna().unique()
    if len(unique_vals) <= 2 and set(unique_vals).issubset({0, 1, True, False}):
        continue
    
    print(f"\n📊 {col}:")
    print(f"   Unique values: {unique_vals}")
    print(f"   Range: {series.min():.2f} to {series.max():.2f}")

print("\n✅ Numeric value inspection complete.")

### 2.3 Detect Garbage Values

**Results**: Looking at the inspection above, no garbage values detected - data looks clean and safe.

In [ ]:
# Detect garbage values: ±inf, huge magnitudes, string infinity representations
weird_report = []

for col in data.columns:
    s = data[col]
    col_info = {"column": col}
    
    if pd.api.types.is_numeric_dtype(s):
        # Check for infinity and huge values
        is_pos_inf = np.isposinf(s)
        is_neg_inf = np.isneginf(s)
        is_huge = (s.abs() > 1e12) & (~s.isna())

        col_info.update({
            "dtype": str(s.dtype),
            "pos_inf": int(is_pos_inf.sum()),
            "neg_inf": int(is_neg_inf.sum()),
            "huge(|x|>1e12)": int(is_huge.sum())
        })

        if any([col_info["pos_inf"], col_info["neg_inf"], col_info["huge(|x|>1e12)"]]):
            weird_report.append(col_info)

    elif s.dtype == "object":
        # Check for string representations of infinity
        s_str = s.astype(str).str.lower().str.strip()
        infinity_tokens = {"inf", "+inf", "-inf", "infinity", "+infinity", "-infinity"}
        inf_str = s_str.isin(infinity_tokens)
        if inf_str.any():
            col_info.update({
                "dtype": str(s.dtype),
                "string_inf": int(inf_str.sum())
            })
            weird_report.append(col_info)

if weird_report:
    print("⚠️ Garbage values detected:")
    display(pd.DataFrame(weird_report).sort_values("column"))
else:
    print("✅ No garbage values detected - data is clean!")

## 3. Detect outliers

We use a conservative approach to detect outliers:
- Determine numerical features (skip binary or features with <=5 differnt values)
- Visually assess statistical characteristics: min, max, average, etc. 

### 3.1 Identify Numeric Columns for Outlier Detection

In [ ]:
# Get numeric columns (excluding ID, binary/flag, and categorical columns)
numeric_cols_for_outliers = []

print("🔍 Identifying numeric columns for outlier detection...")

for col in data.columns:
    col_lower = col.lower()
    series = data[col]
    
    # Skip ID columns, object/categorical columns, datetime columns
    if ("id" in col_lower or 
        series.dtype == "object" or 
        pd.api.types.is_datetime64_any_dtype(series)):
        continue
    
    # Skip binary/flag columns (only 0/1 values)
    unique_vals = series.dropna().unique()
    if len(unique_vals) <= 2 and set(unique_vals).issubset({0, 1, True, False}):
        continue
        
    # Skip columns with very few unique values (likely categorical)
    if series.nunique() <= 5:
        continue
        
    numeric_cols_for_outliers.append(col)

print(f"📊 Found {len(numeric_cols_for_outliers)} numeric columns for outlier detection:")
for col in numeric_cols_for_outliers:
    print(f"   • {col}")

### 3.2 Statistical Summary of Numeric Features

In [ ]:
# Get statistical summary for numeric columns
print("📈 Statistical summary of numeric features:")
print("=" * 80)

for col in numeric_cols_for_outliers:
    series = data[col].dropna()
    if len(series) == 0:
        continue
        
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers_iqr = series[(series < lower_bound) | (series > upper_bound)]
    outlier_pct = (len(outliers_iqr) / len(series)) * 100
    
    print(f"\n📊 {col}:")
    print(f"   Count: {len(series):,}")
    print(f"   Mean: {series.mean():.2f}")
    print(f"   Std: {series.std():.2f}")
    print(f"   Min: {series.min():.2f}")
    print(f"   Q1: {q1:.2f}")
    print(f"   Median: {series.median():.2f}")
    print(f"   Q3: {q3:.2f}")
    print(f"   Max: {series.max():.2f}")
    print(f"   IQR outliers: {len(outliers_iqr):,} ({outlier_pct:.1f}%)")
    print(f"   IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")

print("\n" + "=" * 80)

## 4. Data Type Standardization

### 4.1 Convert Date Columns

In [ ]:
# Convert known date columns to datetime64
date_cols = [
    'dropout_date'
]

print("📅 Converting date columns...")
print("Before conversion:")
display(data[date_cols].dtypes.to_frame("dtype"))

# Convert to datetime
for col in date_cols:
    data[col] = pd.to_datetime(data[col], errors='coerce')

print("\nAfter conversion:")
display(data[date_cols].dtypes.to_frame("dtype"))
print(f"✅ Converted {len(date_cols)} date columns")

### 4.2 Convert Categorical Columns

In [ ]:
# Convert categorical columns to object dtype for later encoding
categorical_cols = [
    'sdo5_degree_COLLEGEJAAR',
    'sdo5_degree_degree',
    'sdo1_previous_Previous_Education_Type',
    'sdo2_skc_ADVIES_DEF',
    'sdo2_orientation_Event_Types_Attended',
    'age_category'
]

print("🏷️ Converting categorical columns to object dtype...")

# Cast to object dtype
for col in categorical_cols:
    data[col] = data[col].astype('object')

# Show value counts for inspection
for col in categorical_cols:
    print(f"\n📊 {col}:")
    counts = data[col].value_counts(dropna=False)
    print(f"   Categories: {len(counts)}")
    print(f"   Top values: {dict(counts.head(3))}")

print(f"\n✅ Converted {len(categorical_cols)} categorical columns")

In [ ]:
# Verify categorical columns with >2 categories
obj_cols = data.select_dtypes(include='object').columns
multi_category_cols = []

for col in obj_cols:
    unique_count = data[col].nunique(dropna=False)
    if unique_count > 2:
        multi_category_cols.append(col)

print(f"📋 Multi-category object columns ({len(multi_category_cols)}):")
for col in multi_category_cols:
    print(f"   • {col}")

### 3.3 Standardize Gender Encoding

In [ ]:
# Convert gender codes to binary: M → 1, V → 0
print("⚥ Standardizing gender encoding...")

before = data['sdo1_characteristics_gender'].value_counts(dropna=False)
print("Before:")
print(before)

gender_map = {"M": 1, "V": 0}
data['sdo1_characteristics_gender'] = (
    data['sdo1_characteristics_gender']
    .map(gender_map)
    .astype('Int64')  # nullable integer type
)

after = data['sdo1_characteristics_gender'].value_counts(dropna=False)
print("\nAfter:")
print(after)
print("✅ Gender encoding standardized")

### 3.4 Standardize Binary/Flag Columns

In [ ]:
# Force all binary/flag fields to 0/1 integer (Int64) dtype
binary_cols = [
    # Dropout flags
    'sdo5_degree_drop_out',
    
    # Student characteristics
    'sdo5_degree_POSTAL_COUNTRY_NL',
    'sdo1_previous_Multiple_Previous_Education',
    'sdo1_characteristics_Dutch_nationality',
    'sdo1_characteristics_gender',
    
    # Academic flags
    'sdo6_results_Enrolled_for_at_least_one_exam',
    'sdo6_results_Average_Grade_A_Sat_Exam',
    'Multiple_Degrees_Enrollment',
    
    # Participation flags
    'has_previous', 'has_skc', 'has_orientation', 'has_dsa', 'has_nf',
    
    # Missing value flags
    'sdo2_orientation_Number_of_Event_Types_missing_flag',
    'sdo2_orientation_Number_of_Events_Attended_missing_flag',
    'sdo2_orientation_First_Event_Date_missing_flag',
    'sdo2_orientation_Last_Event_Date_missing_flag',
    'sdo1_previous_Final_Exam_Date_missing_flag',
    'sdo6_results_Percentage_Credits_A_missing_flag',
    'sdo6_results_Potential_Credits_A_missing_flag',
    'sdo1_prev_distance_distance_to_3584_CS_missing_flag',
    'sdo1_postal_distance_distance_to_3584_CS_missing_flag',
    'time_first_orient_to_start_weeks_missing_flag',
    'time_last_orient_to_start_weeks_missing_flag',
    'gap_skc_to_start_weeks_missing_flag'
]

print(f"🔢 Converting {len(binary_cols)} binary/flag columns to Int64...")

for col in binary_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce').astype('Int64')

print("✅ Binary/flag columns standardized")

### 4.5 Standardize Continuous Numeric Columns

In [ ]:
# Convert continuous numeric measurement columns to float
numeric_float_cols = [
    # Orientation metrics
    'sdo2_orientation_Number_of_Events_Attended',
    'sdo2_orientation_Number_of_Event_Types',
    
    # Academic performance
    'sdo6_results_Total_Credits_Block_A',
    'sdo6_results_Potential_Credits_A',
    'sdo6_results_Percentage_Credits_A',
    'sdo6_results_Average_Grade_A',
    
    # Distance measures
    'sdo1_prev_distance_distance_to_3584_CS',
    'sdo1_postal_distance_distance_to_3584_CS',
    'change_in_distance',
    
    # Time-based features (weeks)
    'time_first_orient_to_start_weeks',
    'time_last_orient_to_start_weeks',
    'gap_skc_to_start_weeks',
    'time_enroll_to_start_weeks',
    'gap_prev_exam_to_start_weeks'
]

print(f"📊 Converting {len(numeric_float_cols)} continuous columns to float...")

for col in numeric_float_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce').astype(float)

print("✅ Continuous numeric columns standardized")

## 5. Save Standardized Dataset

In [ ]:
# Save standardized dataset
output_path = OUT_DIR / "features_standardized.csv"
data.to_csv(output_path, index=False)

print("✅ Data standardization complete!")
print(f"   📂 Saved: {output_path}")
print(f"   📊 Shape: {data.shape}")
print(f"   🏷️ Columns: {len(data.columns)}")

# Show data type summary
print(f"\n📋 Data type summary:")
dtype_counts = data.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"   {dtype}: {count} columns")